In [2]:
from docx import Document
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
import subprocess
from docx.shared import Pt
import pandas as pd
import numpy as np
import os
import shutil
from tqdm import tqdm
import glob
from zipfile import ZipFile 
import concurrent
pd.options.mode.chained_assignment = None 
import json
from docx.shared import RGBColor

In [130]:
df = pd.read_excel('data_sample_generated.xlsx')

In [131]:
df['price_qty'] = df['price_qty'].round(0)
df['vat']= df['price_qty'].round(0)
df['amount']= df['price_qty'].round(0)
df

,company_name,invoice_date,invoice_no,description,qty,price_qty,amount,vat,po,address_1,...,address_3,address_4,address_5,payment_notes,payment_notes_2,bank_account_name,bank,bank_account_number,bank_branch,swift_code
0,PT SUITMEDIA KREASI INDONESIA,2026-04-06 00:00:00,INVGDTI50460,Consultation GP,15,27000.0,27000.0,27000.0,NaN,Gedung Plaza City View Lt. 2,...,"Kelurahan Pejaten Barat, Kecamatan Pasar Minggu",Jakarta Selatan,Indonesia 12510,1. Term of Payment 30 days,NaN,PT Good Doctor Technology Indonesia,BANK OCBC NISP,577800006618,Bank OCBC NISP Cab Rawamangun,NISPIDJA
1,PT SUITMEDIA KREASI INDONESIA,2026-04-06 00:00:00,INVGDTI50460,Consultation SP,8,100000.0,100000.0,100000.0,NaN,Gedung Plaza City View Lt. 3,...,"Kelurahan Pejaten Barat, Kecamatan Pasar Minggu",Jakarta Selatan,Indonesia 12511,1. Term of Payment 30 days,NaN,PT Good Doctor Technology Indonesia,BANK OCBC NISP,577800006618,Bank OCBC NISP Cab Rawamangun,NISPIDJA
2,PT SUITMEDIA KREASI INDONESIA,2026-04-06 00:00:00,INVGDTI50460,Reimbursement of drug purchase,1,5799435.0,5799435.0,5799435.0,NaN,Gedung Plaza City View Lt. 4,...,"Kelurahan Pejaten Barat, Kecamatan Pasar Minggu",Jakarta Selatan,Indonesia 12512,1. Term of Payment 30 days,NaN,PT Good Doctor Technology Indonesia,BANK OCBC NISP,577800006618,Bank OCBC NISP Cab Rawamangun,NISPIDJA
3,PT 123,2026-04-07 00:00:00,INVGDTI50461,Consultation GP,15,27000.0,27000.0,27000.0,NaN,Gedung Plaza City View Lt. 2,...,"Kelurahan Pejaten Barat, Kecamatan Pasar Minggu",Jakarta Selatan,Indonesia 12510,1. Term of Payment 30 days,NaN,PT Good Doctor Technology Indonesia,BANK OCBC NISP,577800006618,Bank OCBC NISP Cab Rawamangun,NISPIDJA
4,PT 123,2026-04-07 00:00:00,INVGDTI50461,Consultation SP,8,100000.0,100000.0,100000.0,NaN,Gedung Plaza City View Lt. 3,...,"Kelurahan Pejaten Barat, Kecamatan Pasar Minggu",Jakarta Selatan,Indonesia 12511,1. Term of Payment 30 days,NaN,PT Good Doctor Technology Indonesia,BANK OCBC NISP,577800006618,Bank OCBC NISP Cab Rawamangun,NISPIDJA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,PT 123,2026-04-18,INVGDTI50490,Specialist Referral Fee,14,75000.0,75000.0,75000.0,NaN,Gedung Plaza City View Lt. 2,...,"Kelurahan Pejaten Barat, Kecamatan Pasar Minggu",Jakarta Selatan,Indonesia 12510,1. Term of Payment 30 days,NaN,PT Good Doctor Technology Indonesia,BANK OCBC NISP,577800006618,Bank OCBC NISP Cab Rawamangun,NISPIDJA
70,PT 123,2026-04-18,INVGDTI50490,Telemedicine Session,15,50000.0,50000.0,50000.0,NaN,Gedung Plaza City View Lt. 2,...,"Kelurahan Pejaten Barat, Kecamatan Pasar Minggu",Jakarta Selatan,Indonesia 12510,1. Term of Payment 30 days,NaN,PT Good Doctor Technology Indonesia,BANK OCBC NISP,577800006618,Bank OCBC NISP Cab Rawamangun,NISPIDJA
71,PT 123,2026-04-18,INVGDTI50490,Home Visit Service,2,500000.0,500000.0,500000.0,NaN,Gedung Plaza City View Lt. 2,...,"Kelurahan Pejaten Barat, Kecamatan Pasar Minggu",Jakarta Selatan,Indonesia 12510,1. Term of Payment 30 days,NaN,PT Good Doctor Technology Indonesia,BANK OCBC NISP,577800006618,Bank OCBC NISP Cab Rawamangun,NISPIDJA
72,PT 123,2026-05-19,INVGDTI50491,Telemedicine Session,8,50000.0,50000.0,50000.0,NaN,Gedung Plaza City View Lt. 2,...,"Kelurahan Pejaten Barat, Kecamatan Pasar Minggu",Jakarta Selatan,Indonesia 12510,1. Term of Payment 30 days,NaN,PT Good Doctor Technology Indonesia,BANK OCBC NISP,577800006618,Bank OCBC NISP Cab Rawamangun,NISPIDJA


In [132]:
df['invoice_date'] = pd.to_datetime(df['invoice_date']).dt.strftime('%d/%m/%Y')



data = df[df['invoice_no'] == 'INVGDTI50460'].fillna('').to_dict(orient = 'records')
data

[{'company_name': 'PT SUITMEDIA KREASI INDONESIA',
  'invoice_date': '06/04/2026',
  'invoice_no': 'INVGDTI50460',
  'description': 'Consultation GP',
  'qty': 15,
  'price_qty': 27000.0,
  'amount': 27000.0,
  'vat': 27000.0,
  'po': '',
  'address_1': 'Gedung Plaza City View Lt. 2',
  'address_2': 'Jl. Kemang Timur No. 22, RT. 001/RW. 008',
  'address_3': 'Kelurahan Pejaten Barat, Kecamatan Pasar Minggu',
  'address_4': 'Jakarta Selatan',
  'address_5': 'Indonesia 12510',
  'payment_notes': '1. Term of Payment 30 days',
  'payment_notes_2': '',
  'bank_account_name': 'PT Good Doctor Technology Indonesia',
  'bank': 'BANK OCBC NISP',
  'bank_account_number': 577800006618,
  'bank_branch': 'Bank OCBC NISP Cab Rawamangun',
  'swift_code': 'NISPIDJA'},
 {'company_name': 'PT SUITMEDIA KREASI INDONESIA',
  'invoice_date': '06/04/2026',
  'invoice_no': 'INVGDTI50460',
  'description': 'Consultation SP',
  'qty': 8,
  'price_qty': 100000.0,
  'amount': 100000.0,
  'vat': 100000.0,
  'po': ''

In [133]:
from docx.shared import Pt
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
from docx.enum.text import WD_ALIGN_PARAGRAPH
from tqdm import tqdm

def set_cell_text(cell, text, font_size=9, font_name='Arial', font_color=None, bold=False, align=None):
    """Set cell text with font formatting."""
    cell.text = (text or '').strip()
    for paragraph in cell.paragraphs:
        pPr = paragraph._p.get_or_add_pPr()
        spacing = OxmlElement('w:spacing')
        spacing.set(qn('w:after'), '0')
        spacing.set(qn('w:before'), '0')
        pPr.append(spacing)

        if align:
            paragraph.alignment = align  # 👈

        for run in paragraph.runs:
            run.font.size = Pt(font_size)
            run.font.name = font_name
            run.font.bold = bold
            if font_color:
                run.font.color.rgb = font_color
                
for i in tqdm(df.invoice_no.unique(), total = df.invoice_no.nunique()):
    data = df[df['invoice_no'] == i].fillna('').to_dict(orient = 'records')
    f = open('Finance Invoice template.docx', 'rb')
    
    
                    
    doc = Document(f)
    
    # Dynamic rows — line items
    subtotal = sum([x.get('amount', 0) for x in data])
    vat = sum([x.get('vat', 0) for x in data])
    total = f"Rp. {(round(subtotal + vat)):,}".replace('.0', '')
    
    subtotal = f"Rp. {round(subtotal):,}".replace('.0', '')
    vat =f"Rp. {round(vat):,}".replace('.0', '')
    
        
    inv_no = data[0].get('invoice_no')
    # Static fields
    static_cells = [
        (1, 4, data[0].get('invoice_date')),
        (2, 4, inv_no),
        (3, 4, data[0].get('po')),
        (14, 0, data[0].get('company_name')),
        (15, 0, data[0].get('address_1')),
        (16, 0, data[0].get('address_2')),
        (17, 0, data[0].get('address_3')),
        (18, 0, data[0].get('address_4')),
        (19, 0, data[0].get('address_5')),
        # (27, 4, str(subtotal).replace('.0', '')),
        # (28, 4, str(vat).replace('.0', '')),
        # (29, 4, str(total).replace('.0', '')),  # fixed: was overwriting row 28 twice
        (32, 0, data[0].get('payment_notes')),
        (36, 0, 'Account Name           : ' + str(data[0].get('bank_account_name', '')).replace('.0', '')),
        (37, 0, 'Bank                          : ' + str(data[0].get('bank', ''))),
        (38, 0, 'Account Number        : ' + str(data[0].get('bank_account_number', ''))),
        (39, 0, 'Branch                       : ' + str(data[0].get('bank_branch', ''))),
        (40, 0, 'Swift Code                 : ' + str(data[0].get('swift_code', ''))),  # fixed: was overwriting row 39
    ]
    
    for row, col, value in static_cells:
        set_cell_text(doc.tables[0].cell(row, col), value)
    
    # Bold cells — subtotal, vat, total
    set_cell_text(doc.tables[0].cell(27, 4), str(subtotal).replace('.0', ''), bold=True)
    set_cell_text(doc.tables[0].cell(28, 4), str(vat).replace('.0', ''),      bold=True)
    set_cell_text(doc.tables[0].cell(29, 4), str(total).replace('.0', ''),    bold=True)
    
    set_cell_text(doc.tables[0].cell(27, 3), "Subtotal", bold=True, align=WD_ALIGN_PARAGRAPH.RIGHT)
    set_cell_text(doc.tables[0].cell(28, 3), "VAT Total",      bold=True, align=WD_ALIGN_PARAGRAPH.RIGHT)
    set_cell_text(doc.tables[0].cell(29, 3), "Total",    bold=True, align=WD_ALIGN_PARAGRAPH.RIGHT)
    
    for i, item in enumerate(data):
        set_cell_text(doc.tables[0].cell(22 + i, 0), item.get('description'))
        set_cell_text(doc.tables[0].cell(22 + i, 2), f"{item.get('price_qty', ''):,}".replace('.0', ''), align=WD_ALIGN_PARAGRAPH.RIGHT)
        set_cell_text(doc.tables[0].cell(22 + i, 3), str(item.get('qty', '')).replace('.0', ''), align=WD_ALIGN_PARAGRAPH.RIGHT)
        set_cell_text(doc.tables[0].cell(22 + i, 4), f"{float(item.get('amount', '')):,}".replace('.0', ''), align=WD_ALIGN_PARAGRAPH.RIGHT)
    
    
    if not os.path.exists(f'output'):
        os.makedirs(f'output')
    
    tmp_name = f'output/Invoice_{inv_no}.docx'
    doc.save(tmp_name)
    f.close()
    subprocess.run(['libreoffice', '--convert-to', 'pdf' ,
                        tmp_name, '--outdir', 'output']
                       ,stdout=subprocess.DEVNULL,
                        stderr=subprocess.DEVNULL
                    )
        
    os.remove(tmp_name)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 32/32 [01:55<00:00,  3.61s/it]


In [128]:
data

,company_name,invoice_date,invoice_no,description,qty,price_qty,amount,vat,po,address_1,...,address_3,address_4,address_5,payment_notes,payment_notes_2,bank_account_name,bank,bank_account_number,bank_brancj,swift_code
0,PT SUITMEDIA KREASI INDONESIA,04/06/2026,INVGDTI50460,Consultation GP,15,27000.0,27000.0,27000.0,NaN,Gedung Plaza City View Lt. 2,...,"Kelurahan Pejaten Barat, Kecamatan Pasar Minggu",Jakarta Selatan,Indonesia 12510,1. Term of Payment 30 days,NaN,PT Good Doctor Technology Indonesia,BANK OCBC NISP,577800006618,Bank OCBC NISP Cab Rawamangun,NISPIDJA
1,PT SUITMEDIA KREASI INDONESIA,04/06/2026,INVGDTI50460,Consultation SP,8,100000.0,100000.0,100000.0,NaN,Gedung Plaza City View Lt. 3,...,"Kelurahan Pejaten Barat, Kecamatan Pasar Minggu",Jakarta Selatan,Indonesia 12511,1. Term of Payment 30 days,NaN,PT Good Doctor Technology Indonesia,BANK OCBC NISP,577800006618,Bank OCBC NISP Cab Rawamangun,NISPIDJA
2,PT SUITMEDIA KREASI INDONESIA,04/06/2026,INVGDTI50460,Reimbursement of drug purchase,1,5799435.0,5799435.0,5799435.0,NaN,Gedung Plaza City View Lt. 4,...,"Kelurahan Pejaten Barat, Kecamatan Pasar Minggu",Jakarta Selatan,Indonesia 12512,1. Term of Payment 30 days,NaN,PT Good Doctor Technology Indonesia,BANK OCBC NISP,577800006618,Bank OCBC NISP Cab Rawamangun,NISPIDJA


In [118]:
subtotal = sum([x.get('amount', 0) for x in data])
vat = sum([x.get('vat', 0) for x in data])
total = f"{(round(subtotal + vat)):,}".replace('.0', '')

In [119]:
total

'11,852,870'

In [24]:
doc = Document(f)

In [56]:
#date
doc.tables[0].cell(1, 4).text = data[0].get('invoice_date')

# invoice
doc.tables[0].cell(2, 4).text = data[0].get('invoice_no')

#PO
doc.tables[0].cell(3, 4).text = data[0].get('po')

#company
doc.tables[0].cell(14, 0).text = data[0].get('company_name')

#address 1
doc.tables[0].cell(15, 0).text = data[0].get('address_1')

#address 2
doc.tables[0].cell(16, 0).text = data[0].get('address_2')

#address 3
doc.tables[0].cell(17, 0).text = data[0].get('address_3')

#address 4
doc.tables[0].cell(18, 0).text = data[0].get('address_4')

#address 5
doc.tables[0].cell(19, 0).text = data[0].get('address_5')

for i in range(data):
    doc.tables[0].cell(23+i, 0).text = i['description']
    doc.tables[1].cell(23+i, 1).text = i['price_qty']
    doc.tables[1].cell(23+i, 2).text = i['qty']
    doc.tables[1].cell(23+i, 3).text = i['amount']

subtotal = sum([x.get('amount',0) for x in data])
vat = sum([x.get('vat',0) for x in data])
total = subtotal + vat

doc.tables[0].cell(27, 4).text = subtotal
doc.tables[0].cell(28, 4).text = vat
doc.tables[0].cell(28, 4).text = total
doc.tables[0].cell(32, 0).text = data[0].get('payment_notes')
doc.tables[0].cell(36, 0).text = 'Account Name           : ' + data[0].get('bank_account_name').astype(str).replace('.0', '')
doc.tables[0].cell(37, 0).text = 'Bank                          : ' + data[0].get('bank').astype(str).replace('.0', '')
doc.tables[0].cell(38, 0).text = 'Account Number        : ' + data[0].get('bank_account_number').astype(str).replace('.0', '')
doc.tables[0].cell(39, 0).text = 'Branch                       : ' + data[0].get('bank_branch').astype(str).replace('.0', '')
doc.tables[0].cell(39, 0).text = 'Swift Code                 : ' + data[0].get('swift_code').astype(str).replace('.0', '')



'Indonesia 12510'

In [59]:
for i in range(data):
    doc.tables[0].cell(23+i, 0).text = i['description']
    doc.tables[1].cell(23+i, 1).text = i['price_qty']
    doc.tables[1].cell(23+i, 2).text = i['qty']
    doc.tables[1].cell(23+i, 3).text = i['amount']

'Consultation SP'

In [69]:
doc.tables[0].cell(36, 0).text 

'Account Name           : PT Good Doctor Technology Indonesia'

In [47]:
for i in range(20):
    for j in range(20):
        print(i,j, '==>', doc.tables[0].cell(i, j).text)
        

0 0 ==> 

0 1 ==> 
0 2 ==> 
0 3 ==> Invoice
0 4 ==> 
0 5 ==> 
0 6 ==> 
0 7 ==> 
0 8 ==> Date
0 9 ==> 06/04/2026
0 10 ==> 
0 11 ==> 
0 12 ==> 
0 13 ==> Invoice #
0 14 ==> INVGDTI50460
0 15 ==> 
0 16 ==> 
0 17 ==> 
0 18 ==> 
0 19 ==> 
1 0 ==> 
1 1 ==> 
1 2 ==> 
1 3 ==> Date
1 4 ==> 06/04/2026
1 5 ==> 
1 6 ==> 
1 7 ==> 
1 8 ==> Invoice #
1 9 ==> INVGDTI50460
1 10 ==> 
1 11 ==> 
1 12 ==> 
1 13 ==> 
1 14 ==> 
1 15 ==> PT Good Doctor Technology Indonesia
1 16 ==> 
1 17 ==> 
1 18 ==> PO #
1 19 ==> 
2 0 ==> 
2 1 ==> 
2 2 ==> 
2 3 ==> Invoice #
2 4 ==> INVGDTI50460
2 5 ==> 
2 6 ==> 
2 7 ==> 
2 8 ==> 
2 9 ==> 
2 10 ==> PT Good Doctor Technology Indonesia
2 11 ==> 
2 12 ==> 
2 13 ==> PO #
2 14 ==> 
2 15 ==> Gedung Capital Place Lt. 42
2 16 ==> 
2 17 ==> 
2 18 ==> 
2 19 ==> 
3 0 ==> 
3 1 ==> 
3 2 ==> 
3 3 ==> 
3 4 ==> 
3 5 ==> PT Good Doctor Technology Indonesia
3 6 ==> 
3 7 ==> 
3 8 ==> PO #
3 9 ==> 
3 10 ==> Gedung Capital Place Lt. 42
3 11 ==> 
3 12 ==> 
3 13 ==> 
3 14 ==> 
3 15 ==> JL. Jend. G

In [60]:
data

[{'company_name': 'PT SUITMEDIA KREASI INDONESIA',
  'invoice_date': Timestamp('2026-04-06 00:00:00'),
  'invoice_no': 'INVGDTI50460',
  'description': 'Consultation GP',
  'qty': 15,
  'price_qty': 26999.999999999996,
  'amount': 404999.99999999994,
  'vat': 44549.99999999999,
  'po': '',
  'address_1': 'Gedung Plaza City View Lt. 2',
  'address_2': 'Jl. Kemang Timur No. 22, RT. 001/RW. 008',
  'address_3': 'Kelurahan Pejaten Barat, Kecamatan Pasar Minggu',
  'address_4': 'Jakarta Selatan',
  'address_5': 'Indonesia 12510',
  'payment_notes': '1. Term of Payment 30 days',
  'payment_notes_2': '',
  'bank_account_name': 'PT Good Doctor Technology Indonesia',
  'bank': 'BANK OCBC NISP',
  'bank_account_number': 577800006618,
  'bank_brancj': 'Bank OCBC NISP Cab Rawamangun',
  'swift_code': 'NISPIDJA'},
 {'company_name': 'PT SUITMEDIA KREASI INDONESIA',
  'invoice_date': Timestamp('2026-04-06 00:00:00'),
  'invoice_no': 'INVGDTI50460',
  'description': 'Consultation SP',
  'qty': 8,
  'p